# 02 Flatten the Stack

## Your Objective
Flatten a column of nested JSON data by unpacking its list items into individual rows in a table.

For example, turn this order record with 2 nested line items:

![images](../images/4IG6WlaConeYXMTITBkXuOlygQ.png)

Into these two records (1 per line item):



![images](../images/M3CwfRx8vobOP7auXhnHCWDnbE.png)

In [19]:
import pandas as pd
import json

In [20]:
df = pd.read_csv("data/SalesOrders.csv")
df.head()

,order_number,order_date,line_items,fulfillment
0,387005,2016-01-22,"[\r\n {\r\n ""product"": {\r\n ""product...",In store
1,395005,2016-01-30,"[\r\n {\r\n ""product"": {\r\n ""product...",In store
2,423011,2016-02-27,"[\r\n {\r\n ""product"": {\r\n ""product...",In store
3,600006,2016-08-22,"[\r\n {\r\n ""product"": {\r\n ""product...",Online
4,652002,2016-10-13,"[\r\n {\r\n ""product"": {\r\n ""product...",In store


In [21]:
df["line_items"] = df["line_items"].apply(json.loads)

df.head()

,order_number,order_date,line_items,fulfillment
0,387005,2016-01-22,[{'product': {'product_name': 'MGS Age of Empi...,In store
1,395005,2016-01-30,[{'product': {'product_name': 'Contoso DVD 60 ...,In store
2,423011,2016-02-27,[{'product': {'product_name': 'Contoso 16GB Mp...,In store
3,600006,2016-08-22,[{'product': {'product_name': 'Contoso Water H...,Online
4,652002,2016-10-13,[{'product': {'product_name': 'Contoso Genuine...,In store


In [22]:
df = df.explode("line_items")

df.head()

,order_number,order_date,line_items,fulfillment
0,387005,2016-01-22,{'product': {'product_name': 'MGS Age of Empir...,In store
0,387005,2016-01-22,{'product': {'product_name': 'A. Datum Bridge ...,In store
0,387005,2016-01-22,{'product': {'product_name': 'WWI Desktop PC1....,In store
1,395005,2016-01-30,{'product': {'product_name': 'Contoso DVD 60 D...,In store
1,395005,2016-01-30,{'product': {'product_name': 'SV DVD 38 DVD St...,In store


In [25]:
line_items_df = (
    pd.json_normalize(df["line_items"], sep="_")
    .rename({"product_product_name": "product_name", "product_product_price": "product_price"}, axis=1)
)
line_items_df.head()

,quantity,product_name,product_price
0,3,MGS Age of Empires II Gold Edition2009 E172,32.00
0,2,A. Datum Bridge Digital Camera M300 Pink,186.90
0,1,WWI Desktop PC1.80 E1801 Silver,269.90
1,5,Contoso DVD 60 DVD Storage Binder L20 Black,22.89
1,5,SV DVD 38 DVD Storage Binder E25 Silver,9.99


In [29]:
result_df = pd.concat([df.drop(columns=["line_items"]), line_items_df], axis=1)
result_df["rev"] = result_df["product_price"] * result_df["quantity"]

result_df.head()

,order_number,order_date,fulfillment,quantity,product_name,product_price,rev
0,387005,2016-01-22,In store,3,MGS Age of Empires II Gold Edition2009 E172,32.00,96.00
0,387005,2016-01-22,In store,2,A. Datum Bridge Digital Camera M300 Pink,186.90,373.80
0,387005,2016-01-22,In store,1,WWI Desktop PC1.80 E1801 Silver,269.90,269.90
1,395005,2016-01-30,In store,5,Contoso DVD 60 DVD Storage Binder L20 Black,22.89,114.45
1,395005,2016-01-30,In store,5,SV DVD 38 DVD Storage Binder E25 Silver,9.99,49.95


In [32]:
result_df.query("fulfillment=='Online'")["rev"].sum()

np.float64(18238.979999999996)